# Single-Mask DF-DPC — Calculadora de parámetros de laboratorio

Implementa las condiciones geométricas descritas en el paper (Sección EXPERIMENT) y devuelve los rangos permitidos para:

- Distancia fuente–máscara $d_{sm}$
- Distancia máscara–detector $d_{md}$
- Distancia fuente–detector total $L = d_{sm}+d_{md}$ (con tope de laboratorio, por defecto **0.7 m**)
- Distancia muestra–detector $d_{od}$ (muestra apoyada contra la máscara)
- Magnificación geométrica $M$
- Ancho de slit $w$ — variable libre porque tu sistema lo modula con dos máscaras superpuestas
- Pasos de *dithering* (sub-período de máscara)
- Resolución limitada por foco y *pixel size* efectivo a la muestra

### Condiciones del paper
El alineamiento exige que la proyección de la máscara sobre el detector tenga un período entero $N$ veces el pitch del detector:
$$ p_m \cdot M \;=\; N \cdot p_d $$
donde
- $N=2$ → configuración **DPC** o **DF** (M ≈ 2×)
- $N=3$ → configuración **DF-DPC** (M ≈ 3×)

y la magnificación es $M = (d_{sm}+d_{md})/d_{sm}$, lo que fija el cociente entre distancias una vez elegida $N$.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Parámetros de laboratorio

Los defaults coinciden con los del paper. `slit_um=None` deja que la calculadora elija el óptimo; asigná un número para forzar tu valor.

In [ ]:
@dataclass
class LabSetup:
    # --- Máscara ---
    p_mask_um: float = 53.0       # período de la máscara [µm]
    slit_um: Optional[float] = None  # ancho de slit (None → se computa el óptimo)
    au_thickness_um: float = 100. # espesor del oro [µm]
    # Bounds del slit ajustable por superposición de dos máscaras:
    slit_min_um: float = 2.0      # mínimo alcanzable (fabricación / mecánica) [µm]
    slit_max_um: float = 50.0     # máximo alcanzable (< p_mask) [µm]
    slit_safety: float = 0.85     # factor de seguridad sobre el máximo teórico (0<f≤1)

    # --- Detector ---
    p_det_um: float = 55.0        # pixel pitch [µm]
    det_w_mm: float = 70.0        # ancho activo [mm]
    det_h_mm: float = 14.0        # alto activo  [mm]

    # --- Fuente ---
    focal_spot_um: float = 20.0   # diámetro nominal del foco [µm] (Hamamatsu: 7/20/50)
    kV: float = 40.0              # tensión del tubo [kV]

    # --- Restricciones del banco ---
    L_max_m: float = 0.70         # distancia fuente-detector máxima [m]
    L_min_m: float = 0.20         # distancia fuente-detector mínima razonable [m]

    # --- Adquisición ---
    config: str = 'DF-DPC'        # 'DPC', 'DF' o 'DF-DPC'
    n_dither: int = 8             # pasos de dithering por período de máscara

lab = LabSetup()
lab

## 2. Geometría según la configuración

Dada $N$ ($=2$ para DPC/DF, $=3$ para DF-DPC) y los pitches, queda fijada la magnificación nominal
$$M^\star = \frac{N\,p_d}{p_m}$$
y, para cualquier $L \le L_{max}$:
$$d_{sm} = \frac{L}{M^\star}, \qquad d_{md} = L\,\frac{M^\star-1}{M^\star}.$$

In [ ]:
_N_BY_CONFIG = {'DPC': 2, 'DF': 2, 'DF-DPC': 3}

def nominal_magnification(lab: LabSetup, config: Optional[str] = None) -> float:
    cfg = (config or lab.config).upper()
    if cfg not in _N_BY_CONFIG:
        raise ValueError(f"config debe ser uno de {list(_N_BY_CONFIG)}")
    return _N_BY_CONFIG[cfg] * lab.p_det_um / lab.p_mask_um

def distances_from_L(L_m, M):
    d_sm = L_m / M
    return d_sm, L_m - d_sm

for cfg in ['DPC', 'DF-DPC']:
    M = nominal_magnification(lab, cfg)
    print(f"{cfg:7s}  N={_N_BY_CONFIG[cfg]}   M* = N·p_d/p_m = {M:.4f}")
    print(f"          p_mask·M* = {lab.p_mask_um*M:.2f} µm  vs  N·p_det = {_N_BY_CONFIG[cfg]*lab.p_det_um:.2f} µm")

## 3. Optimización del ancho de slit

El slit $w$ es libre (sistema de doble máscara). Para que cada beamlet ilumine **un sólo pixel** del detector (contraste pixel-a-pixel — requisito de DPC y DF):

$$ \underbrace{w \cdot M}_{\text{geométrico}} \;+\; \underbrace{f_s\,(M-1)}_{\text{penumbra del foco}} \;\le\; p_d \;\;\Longrightarrow\;\; w_{\max} = \frac{p_d - f_s(M-1)}{M} $$

- **Si $f_s(M-1) \ge p_d$** la penumbra ya cubre un pixel: ningún slit funciona (warning crítico).
- **Óptimo recomendado**: $w^\star = \text{safety} \cdot w_{\max}$, recortado a `[slit_min_um, slit_max_um]` y al duty < 0.9.
- Se chequea además que el paso de dithering ≤ $w^\star$ para muestrear el slit.

In [ ]:
def slit_bounds(lab: LabSetup, config: Optional[str] = None):
    """Devuelve (w_max_teorico, w_opt, warnings_list) en µm."""
    cfg = (config or lab.config).upper()
    M = nominal_magnification(lab, cfg)
    fs, pd, pm = lab.focal_spot_um, lab.p_det_um, lab.p_mask_um
    warns = []
    penumbra_det = fs * (M - 1)
    if penumbra_det >= pd:
        warns.append(
            f"[CRÍTICO] Foco {fs:.1f} µm: penumbra en detector = {penumbra_det:.1f} µm ≥ p_det={pd:.1f} µm "
            f"para {cfg} (M={M:.3f}). No existe slit válido — usá un foco menor "
            f"(< {pd/(M-1):.1f} µm) o cambiá a una configuración con menor M."
        )
        return float('nan'), float('nan'), warns
    w_max_teo = (pd - penumbra_det) / M
    w_opt = lab.slit_safety * w_max_teo
    upper_mech = min(lab.slit_max_um, pm * 0.9)
    if w_opt > upper_mech:
        warns.append(f"[INFO] w_opt teórico ({w_opt:.1f} µm) excede límite mecánico/duty ({upper_mech:.1f} µm); recortado.")
        w_opt = upper_mech
    if w_opt < lab.slit_min_um:
        warns.append(
            f"[ADVERTENCIA] w_opt = {w_opt:.1f} µm < slit_min = {lab.slit_min_um:.1f} µm: "
            f"la doble máscara no puede abrir un slit tan angosto. Señal baja y/o contraste pobre."
        )
        w_opt = lab.slit_min_um
    if w_opt >= pm:
        warns.append(f"[CRÍTICO] w_opt {w_opt:.1f} µm ≥ p_mask {pm:.1f} µm: la máscara queda transparente.")
    step = pm / lab.n_dither
    if step > w_opt:
        warns.append(
            f"[ADVERTENCIA] paso de dithering ({step:.2f} µm) > w_opt ({w_opt:.2f} µm): "
            f"subí n_dither (actual {lab.n_dither}) a ≥ {int(np.ceil(pm/w_opt))}."
        )
    return w_max_teo, w_opt, warns

def resolve_slit(lab: LabSetup, config: Optional[str] = None):
    """Devuelve (w_um, source∈{'user','auto'}, warnings)."""
    _, w_opt, warns = slit_bounds(lab, config)
    if lab.slit_um is not None:
        return lab.slit_um, 'user', warns
    return w_opt, 'auto', warns

def ideal_slit_report(lab: LabSetup):
    print(f"Foco fuente : {lab.focal_spot_um:.1f} µm")
    print(f"p_mask      : {lab.p_mask_um:.1f} µm   p_det : {lab.p_det_um:.1f} µm")
    print(f"slit bounds : [{lab.slit_min_um:.1f}, {lab.slit_max_um:.1f}] µm   safety={lab.slit_safety}")
    print()
    for cfg in ['DPC', 'DF-DPC']:
        M = nominal_magnification(lab, cfg)
        w_max, w_opt, warns = slit_bounds(lab, cfg)
        print(f"--- {cfg}  (M={M:.3f}) ---")
        if np.isnan(w_max):
            print(f"  w_max (teórico) : N/A")
            print(f"  w_opt           : N/A")
        else:
            print(f"  w_max (teórico) : {w_max:6.2f} µm   (beamlet = 1 px)")
            print(f"  w_opt recomend. : {w_opt:6.2f} µm   (× safety {lab.slit_safety})")
            print(f"  zona iluminada en detector: {w_opt*M + lab.focal_spot_um*(M-1):5.2f} µm "
                  f"(geom {w_opt*M:.2f} + penumbra {lab.focal_spot_um*(M-1):.2f})")
            print(f"  duty cycle (w/p_mask) : {w_opt/lab.p_mask_um:.3f}")
        for w in warns:
            warnings.warn(w)
            print(f"  {w}")
        print()

ideal_slit_report(lab)

In [ ]:
def plot_slit_design(lab: LabSetup):
    """Zona iluminada en detector vs ancho de slit, para los 3 focos del Hamamatsu."""
    focales = [7.0, 20.0, 50.0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, cfg in zip(axes, ['DPC', 'DF-DPC']):
        M = nominal_magnification(lab, cfg)
        w_range = np.linspace(0.5, min(lab.slit_max_um, lab.p_mask_um*0.95), 200)
        for fs in focales:
            zone = w_range * M + fs * (M - 1)
            line, = ax.plot(w_range, zone, label=f'foco {fs:.0f} µm')
            if fs * (M-1) < lab.p_det_um:
                w_max = (lab.p_det_um - fs*(M-1)) / M
                ax.axvline(w_max, color=line.get_color(), ls=':', alpha=0.6)
        ax.axhline(lab.p_det_um, color='red', ls='--', label=f'p_det={lab.p_det_um:.0f} µm')
        ax.axvspan(lab.slit_min_um, lab.slit_max_um, color='gray', alpha=0.08,
                   label=f'slit alcanzable [{lab.slit_min_um:.0f},{lab.slit_max_um:.0f}]')
        _, w_opt, _ = slit_bounds(lab, cfg)
        if not np.isnan(w_opt):
            ax.axvline(w_opt, color='black', lw=2, label=f'w_opt({lab.focal_spot_um:.0f} µm)={w_opt:.1f} µm')
        ax.set_xlabel('ancho de slit  w  [µm]')
        ax.set_ylabel('zona iluminada en detector  [µm]')
        ax.set_title(f"{cfg}   M={M:.3f}")
        ax.grid(alpha=.3); ax.legend(fontsize=7, loc='upper left')
    fig.suptitle('Diseño del slit — el beamlet debe caber en 1 pixel')
    fig.tight_layout()
    return fig

plot_slit_design(lab);

## 4. Rangos permitidos de distancias

Barrido de $L$ entre $L_{min}$ y $L_{max}$, manteniendo $p_m M = N p_d$.

In [ ]:
def sweep_geometry(lab: LabSetup, config: Optional[str] = None, n: int = 200):
    M = nominal_magnification(lab, config)
    L = np.linspace(lab.L_min_m, lab.L_max_m, n)
    d_sm, d_md = distances_from_L(L, M)
    d_od = d_md  # muestra contra la máscara
    M_obj = (d_sm + d_od) / d_sm
    px_eff_um = lab.p_det_um / M_obj
    blur_det_um = lab.focal_spot_um * d_od / d_sm
    blur_obj_um = blur_det_um / M_obj
    return dict(L=L, d_sm=d_sm, d_md=d_md, d_od=d_od, M=np.full_like(L, M),
                M_obj=M_obj, px_eff_um=px_eff_um,
                blur_det_um=blur_det_um, blur_obj_um=blur_obj_um,
                fov_w_mm=lab.det_w_mm/M_obj, fov_h_mm=lab.det_h_mm/M_obj)

def summary_table(lab: LabSetup):
    rows = []
    for cfg in ['DPC', 'DF-DPC']:
        s = sweep_geometry(lab, cfg, n=2)
        _, w_opt, _ = slit_bounds(lab, cfg)
        w_show = lab.slit_um if lab.slit_um is not None else w_opt
        for i, tag in enumerate(['min', 'max']):
            rows.append([
                cfg, tag,
                f"{s['L'][i]*100:.1f}", f"{s['d_sm'][i]*100:.1f}",
                f"{s['d_md'][i]*100:.1f}", f"{s['M'][i]:.3f}",
                f"{s['px_eff_um'][i]:.2f}", f"{s['blur_obj_um'][i]:.2f}",
                f"{s['fov_w_mm'][i]:.1f} × {s['fov_h_mm'][i]:.1f}",
                f"{w_show:.2f}" if not np.isnan(w_show) else 'N/A',
            ])
    hdr = ['cfg','L', 'L[cm]', 'd_sm[cm]', 'd_md[cm]', 'M', 'px_eff[µm]', 'blur_obj[µm]', 'FOV[mm]', 'slit[µm]']
    width = [max(len(h), max(len(r[i]) for r in rows)) for i, h in enumerate(hdr)]
    line = '  '.join(h.ljust(width[i]) for i, h in enumerate(hdr))
    print(line); print('-'*len(line))
    for r in rows:
        print('  '.join(str(c).ljust(width[i]) for i,c in enumerate(r)))

summary_table(lab)

## 5. Punto de operación

Dada $L$ (m), devuelve todos los parámetros y emite warnings si la configuración no es físicamente realizable.

In [ ]:
def operating_point(lab: LabSetup, L_m: float, config: Optional[str] = None):
    cfg = (config or lab.config).upper()
    if L_m > lab.L_max_m + 1e-9:
        raise ValueError(f"L={L_m*100:.1f} cm excede el tope ({lab.L_max_m*100:.1f} cm)")
    if L_m < lab.L_min_m:
        raise ValueError(f"L={L_m*100:.1f} cm por debajo del mínimo ({lab.L_min_m*100:.1f} cm)")
    M = nominal_magnification(lab, cfg)
    d_sm, d_md = distances_from_L(L_m, M)
    d_od = d_md
    M_obj = (d_sm + d_od) / d_sm
    px_eff_um = lab.p_det_um / M_obj
    blur_det_um = lab.focal_spot_um * d_od / d_sm
    blur_obj_um = blur_det_um / M_obj
    w_um, w_src, w_warns = resolve_slit(lab, cfg)
    illum_det_um = (w_um * M + blur_det_um) if (w_um and not np.isnan(w_um)) else float('nan')
    step_mask_um = lab.p_mask_um / lab.n_dither
    step_det_um  = step_mask_um * M
    print(f"Configuración: {cfg}   (N = {_N_BY_CONFIG[cfg]})")
    print(f"  L (fuente-detector)  : {L_m*100:6.2f} cm   [tope: {lab.L_max_m*100:.1f} cm]")
    print(f"  d_sm (fuente-máscara): {d_sm*100:6.2f} cm")
    print(f"  d_md (máscara-det.)  : {d_md*100:6.2f} cm")
    print(f"  d_od (muestra-det.)  : {d_od*100:6.2f} cm   (muestra contra máscara)")
    print(f"  Magnificación M      : {M:.4f}")
    print(f"  Período proyectado   : {lab.p_mask_um*M:6.2f} µm  ({_N_BY_CONFIG[cfg]} × p_det = {_N_BY_CONFIG[cfg]*lab.p_det_um:.2f} µm)")
    print()
    print(f"  Slit ({w_src})        : {w_um:6.2f} µm   (duty={w_um/lab.p_mask_um:.3f})" if not np.isnan(w_um) else "  Slit                : N/A")
    if not np.isnan(illum_det_um):
        print(f"  Zona iluminada en det.: {illum_det_um:6.2f} µm   (vs p_det={lab.p_det_um:.1f} µm)")
        if illum_det_um > lab.p_det_um:
            msg = f"beamlet ({illum_det_um:.2f} µm) excede p_det ({lab.p_det_um:.1f} µm) — pérdida de contraste pixel-a-pixel"
            warnings.warn(msg)
            print(f"  ⚠  {msg}")
    print()
    print(f"  Pixel efectivo en muestra : {px_eff_um:6.2f} µm/px")
    print(f"  Penumbra (foco {lab.focal_spot_um:.0f} µm):")
    print(f"       en detector  : {blur_det_um:6.2f} µm")
    print(f"       en muestra   : {blur_obj_um:6.2f} µm")
    print(f"  FOV en muestra   : {lab.det_w_mm/M_obj:5.1f} × {lab.det_h_mm/M_obj:5.1f} mm")
    print()
    print(f"  Dithering ({lab.n_dither} pasos / período):")
    print(f"       paso en máscara/muestra : {step_mask_um:6.2f} µm")
    print(f"       paso equivalente en det.: {step_det_um:6.2f} µm")
    for w in w_warns:
        warnings.warn(w)
        print(f"  {w}")
    return dict(cfg=cfg, M=M, d_sm=d_sm, d_md=d_md, d_od=d_od,
                slit_um=w_um, slit_source=w_src, illum_det_um=illum_det_um,
                px_eff_um=px_eff_um, blur_obj_um=blur_obj_um,
                step_mask_um=step_mask_um, step_det_um=step_det_um,
                warnings=w_warns)

_ = operating_point(lab, L_m=0.65, config='DF-DPC')

## 6. Visualización del barrido

In [ ]:
def plot_sweep(lab: LabSetup):
    fig, ax = plt.subplots(2, 2, figsize=(11, 7.5))
    for cfg, color in [('DPC', 'tab:blue'), ('DF-DPC', 'tab:orange')]:
        s = sweep_geometry(lab, cfg, n=200)
        L_cm = s['L']*100
        ax[0,0].plot(L_cm, s['d_sm']*100, color=color, label=f"{cfg}  d_sm")
        ax[0,0].plot(L_cm, s['d_md']*100, color=color, ls='--', label=f"{cfg}  d_md")
        ax[0,1].plot(L_cm, s['M'], color=color, label=f"{cfg}  M = {s['M'][0]:.3f}")
        ax[1,0].plot(L_cm, s['px_eff_um'], color=color, label=cfg)
        ax[1,1].plot(L_cm, s['blur_obj_um'], color=color, label=cfg)
    for a in ax.ravel():
        a.set_xlabel('L = d_sm + d_md  [cm]'); a.grid(alpha=.3); a.legend(fontsize=8)
    ax[0,0].set_ylabel('distancia [cm]'); ax[0,0].set_title('Distancias fuente-máscara / máscara-det.')
    ax[0,1].set_ylabel('M'); ax[0,1].set_title('Magnificación nominal')
    ax[1,0].set_ylabel('µm/px en muestra'); ax[1,0].set_title('Pixel efectivo (p_det / M)')
    ax[1,1].set_ylabel('µm'); ax[1,1].set_title(f"Penumbra en muestra (foco {lab.focal_spot_um:.0f} µm)")
    fig.suptitle(f"Single-Mask DF-DPC  |  p_mask={lab.p_mask_um} µm, p_det={lab.p_det_um} µm, L_max={lab.L_max_m*100:.0f} cm")
    fig.tight_layout()
    return fig

plot_sweep(lab);

## 7. Diagrama de la geometría seleccionada

In [ ]:
def plot_geometry(lab: LabSetup, L_m: float, config: Optional[str] = None):
    cfg = (config or lab.config).upper()
    M = nominal_magnification(lab, cfg)
    d_sm, d_md = distances_from_L(L_m, M)
    w_um, w_src, _ = resolve_slit(lab, cfg)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    y0 = 0
    ax.hlines(y0, 0, L_m*100, color='lightgray', lw=1)
    ax.plot(0, y0, 'o', color='gold', markersize=14, markeredgecolor='k')
    ax.text(0, y0+0.18, f"Fuente\n({lab.focal_spot_um:.0f} µm, {lab.kV:.0f} kV)", ha='center', fontsize=8)
    xm = d_sm*100
    ax.vlines(xm, y0-0.12, y0+0.12, color='tab:orange', lw=4)
    slit_lbl = f"{w_um:.1f} µm ({w_src})" if not np.isnan(w_um) else 'N/A'
    ax.text(xm, y0+0.18, f"Máscara\np={lab.p_mask_um:.0f} µm\nslit={slit_lbl}", ha='center', fontsize=8)
    ax.vlines(xm+0.4, y0-0.08, y0+0.08, color='tab:green', lw=3)
    ax.text(xm+0.4, y0-0.28, 'Muestra', ha='center', fontsize=8, color='tab:green')
    xd = (d_sm+d_md)*100
    ax.vlines(xd, y0-0.15, y0+0.15, color='tab:blue', lw=5)
    ax.text(xd, y0+0.21, f"Detector\np={lab.p_det_um:.0f} µm", ha='center', fontsize=8)
    ax.annotate('', xy=(xm, y0-0.35), xytext=(0, y0-0.35), arrowprops=dict(arrowstyle='<->'))
    ax.text(xm/2, y0-0.42, f"d_sm = {d_sm*100:.1f} cm", ha='center', fontsize=9)
    ax.annotate('', xy=(xd, y0-0.35), xytext=(xm, y0-0.35), arrowprops=dict(arrowstyle='<->'))
    ax.text((xm+xd)/2, y0-0.42, f"d_md = {d_md*100:.1f} cm", ha='center', fontsize=9)
    ax.annotate('', xy=(xd, y0+0.55), xytext=(0, y0+0.55), arrowprops=dict(arrowstyle='<->'))
    ax.text(xd/2, y0+0.6, f"L = {L_m*100:.1f} cm   (M = {M:.3f}, {cfg})", ha='center', fontsize=9, weight='bold')
    ax.set_ylim(-0.6, 0.85); ax.set_xlim(-3, L_m*100+3)
    ax.set_yticks([]); ax.set_xlabel('posición a lo largo del eje óptico [cm]')
    ax.set_title(f"Geometría {cfg}")
    fig.tight_layout()
    return fig

plot_geometry(lab, L_m=0.65, config='DF-DPC');

## 8. Validación: ¿la condición de Moiré nulo es alcanzable?

In [ ]:
def check_feasibility(lab: LabSetup, tol: float = 0.15):
    ok = []
    for cfg, N in _N_BY_CONFIG.items():
        if cfg == 'DF':
            continue
        M = nominal_magnification(lab, cfg)
        deviation = abs(M - N) / N
        status = 'OK' if deviation <= tol else 'REVISAR'
        ok.append((cfg, N, M, deviation, status))
        msg = f"{cfg:7s}  M* = {M:.3f}   target N = {N}   desvío = {deviation*100:5.2f}%   → {status}"
        print(msg)
        if status != 'OK':
            warnings.warn(f"{cfg}: desvío {deviation*100:.1f}% del Moiré nulo — ajustá p_mask o p_det")
    print()
    for cfg, _, M, *_ in ok:
        d_sm_max, d_md_max = distances_from_L(lab.L_max_m, M)
        print(f"{cfg:7s}  con L = L_max = {lab.L_max_m*100:.1f} cm  →  d_sm = {d_sm_max*100:.1f} cm, d_md = {d_md_max*100:.1f} cm")

check_feasibility(lab)

## 9. Test rápido con tus valores

Editá la celda. Dejá `slit_um=None` para que se compute el slit óptimo.

In [ ]:
mi_lab = LabSetup(
    p_mask_um   = 53.0,
    slit_um     = None,       # ← None → optimizar; o poné un número [µm] para fijarlo
    slit_min_um = 2.0,
    slit_max_um = 50.0,
    slit_safety = 0.85,
    p_det_um    = 55.0,
    det_w_mm    = 70.0,
    det_h_mm    = 14.0,
    focal_spot_um = 20.0,     # probá 7 / 20 / 50 para ver cómo cambian los warnings
    kV          = 40.0,
    L_max_m     = 0.70,
    L_min_m     = 0.25,
    config      = 'DF-DPC',
    n_dither    = 8,
)

summary_table(mi_lab); print()
check_feasibility(mi_lab); print()
ideal_slit_report(mi_lab)
operating_point(mi_lab, L_m=mi_lab.L_max_m, config='DF-DPC')
plot_sweep(mi_lab)
plot_slit_design(mi_lab)
plot_geometry(mi_lab, L_m=mi_lab.L_max_m, config='DF-DPC');

### Stress test: foco grande
Con foco de 50 µm la penumbra ya cubre un pixel en DF-DPC → ningún slit válido. El sistema debe emitir warnings.

In [ ]:
stress = LabSetup(focal_spot_um=50.0, config='DF-DPC', slit_um=None)
ideal_slit_report(stress)